# 统一配置与主模型结果绘图


读取当前实验选择与新预测：四类数据各五 seed，DLPFC 包含12张切片。
HBC 自动跟随最近成功运行的 paper/refined 方案，不按成绩自动择优。
指标重算、绘图使用同一套当前文件；缺失或不匹配时停止，不回退到历史归档。
图件保存到 `results/figures/current/<数据>/<方案>/`，并在下方直接展开预览。
历史对照方法和消融结果仍在库外补充实验目录，本 Notebook 不将它们伪装为本次重跑结果。


In [ ]:
from pathlib import Path
import sys, importlib.util
ROOT = next((q for p in [Path.cwd(), *Path.cwd().parents]
             for q in [p, p / "sidmgf_final"] if (q / "config/paper.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请在 sidmgf_final 或其 tutorials 目录中打开 Notebook")
spec = importlib.util.spec_from_file_location("spasrd_notebook_output", ROOT / "src/notebook_output.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
RESULTS = module.NotebookResults(ROOT)
execute = RESULTS.execute
print("项目目录:", ROOT)
print("运行日志、指标表和图件预览将直接显示在本 Notebook 中。")


In [ ]:
import json
import pandas as pd
CONFIG=json.loads((ROOT/'config/paper.json').read_text(encoding='utf-8'))
SELECTION=json.loads((ROOT/'results/current_selection.json').read_text(encoding='utf-8'))['datasets']
display(pd.DataFrame([dict(dataset=r['dataset'],sample=r['sample'],donor=r['donor'],
                          profile=r['profile'],checkpoint=SELECTION[r['dataset']]['checkpoint'],**r['parameters'])
                      for r in CONFIG['runs'] if r['seed']==0 and r['profile']==SELECTION[r['dataset']]['profile']]))


In [ ]:
execute("summary", "--source", "current")


In [ ]:
# 自动读取当前实验方案和新模型预测，并保存、显示图件。
execute("figures", "--checkpoint", "current")
